# investalyze: data quality review

Findings from `python -m investalyze.quality`: 17 checks over prices, market_data, dividends,
splits and the fundamentals tables, stored in the `anomalies` table (each run replaces a check's rows).

Triage: confirmed quirk -> entry in `9999_data_quirks.ipynb` -> fix in `cleaning.toml` ->
`python -m investalyze.cleaning apply` -> re-run the checks.

In [1]:
%load_ext autoreload
%autoreload 2
from helpers import connect_readonly, get_anomaly_summary, show_check, show_df

con = connect_readonly()

## Summary

In [2]:
show_df(get_anomaly_summary(con))

,CheckName,Severity,Findings,Tickers,DetectedAt
0,nonpositive_price,error,"21,246",19,2026-07-06
1,ohlc_inconsistent,error,"11,259",361,2026-07-06
2,hard_invariants,error,4,1,2026-07-06
3,fundamentals_sanity,error,4,1,2026-07-06
4,bond_yield_bound,warn,"25,303",45,2026-07-06
5,stale_run,warn,"6,074","1,060",2026-07-06
6,quarters_vs_fy,warn,"5,982","1,531",2026-07-06
7,extreme_return,warn,"5,636","1,707",2026-07-06
8,income_chain,warn,"5,282",183,2026-07-06
9,balance_subtotals,warn,"2,284",162,2026-07-06


## Prices & market

### nonpositive_price

error: any of O/H/L/C/AC <= 0 in `prices`; O/H/L/C <= 0 in `market_data` except bonds (yields).

In [3]:
show_check(con, 'nonpositive_price')

,Ticker,Findings,First,Last
0,VHI,"5,429",1980-03-17,2001-09-07
1,CXE,"3,326",1994-10-27,2008-12-31
2,CMU,"3,323",1994-10-27,2008-12-31
3,VATE,"2,729",2009-07-13,2020-05-13
4,CXH,"2,461",1998-02-17,2008-12-31
5,CBIO,"2,189",2014-01-10,2022-09-20
6,SAFE,"1,024",1989-11-16,1993-12-03
7,DEC,732,2021-01-04,2023-11-29
8,ATLX,15,2015-06-26,2016-12-28
9,RCAT,9,2008-01-31,2012-12-28


,SrcTable,Ticker,Date,Key,Details
0,prices,ATLX,2015-06-26,,O=375.0 H=375.0 L=0.0 C=375.0 AC=375.0
1,prices,ATLX,2015-07-09,,O=375.0 H=375.0 L=0.0 C=375.0 AC=375.0
2,prices,ATLX,2015-07-10,,O=375.0 H=375.0 L=0.0 C=375.0 AC=375.0
3,prices,ATLX,2015-07-15,,O=375.0 H=375.0 L=0.0 C=375.0 AC=375.0
4,prices,ATLX,2015-07-17,,O=375.0 H=375.0 L=0.0 C=375.0 AC=375.0
5,prices,ATLX,2015-07-20,,O=375.0 H=375.0 L=0.0 C=375.0 AC=375.0
6,prices,ATLX,2015-07-21,,O=375.0 H=375.0 L=0.0 C=375.0 AC=375.0
7,prices,ATLX,2015-07-22,,O=0.0 H=375.0 L=0.0 C=375.0 AC=375.0
8,prices,ATLX,2015-07-24,,O=0.0 H=375.0 L=0.0 C=375.0 AC=375.0
9,prices,ATLX,2015-07-27,,O=375.0 H=375.0 L=0.0 C=375.0 AC=375.0


### ohlc_inconsistent

error: H < L, or O/C outside [L, H]; both tables, all asset classes.

In [4]:
show_check(con, 'ohlc_inconsistent')

,Ticker,Findings,First,Last
0,CXE,"3,327",1994-10-27,2023-06-05
1,CMU,"3,323",1994-10-27,2008-12-31
2,CXH,"2,463",1998-02-17,2023-06-05
3,1YCHY,587,2014-12-24,2022-06-02
4,^TSX,338,2012-08-01,2026-06-04
5,^PSEI,224,2016-10-12,2023-07-18
6,^AOR,130,2016-10-17,2026-05-05
7,^TWSE,80,2013-04-10,2026-03-17
8,^JCI,58,2017-05-31,2026-01-19
9,^KLCI,33,2017-01-12,2020-07-16


,SrcTable,Ticker,Date,Key,Details
0,market_data,10YCNY,2013-04-08,,O=3.43 H=3.43 L=3.477 C=3.477
1,market_data,10YCNY,2015-05-25,,O=3.46 H=3.458 L=3.445 C=3.46
2,market_data,10YCZY,2012-11-19,,O=2.047 H=2.062 L=2.102 C=2.102
3,market_data,10YCZY,2014-08-28,,O=1.22 H=1.213 L=1.195 C=1.195
4,market_data,10YKRY,2013-05-09,,O=2.8 H=2.76 L=2.84 C=2.84
5,market_data,10YROY,2013-09-10,,O=5.26 H=5.26 L=5.6 C=5.55
6,market_data,1YCHY,2014-12-24,,O=0.3 H=0.25 L=0.3 C=0.25
7,market_data,1YCHY,2014-12-26,,O=0.31 H=0.25 L=0.31 C=0.25
8,market_data,1YCHY,2014-12-31,,O=0.22 H=0.18 L=0.22 C=0.18
9,market_data,1YCHY,2015-01-02,,O=0.17 H=0.17 L=0.25 C=0.25


### negative_volume

error: V < 0 in `prices`.

In [5]:
show_check(con, 'negative_volume')

*clean*

### bond_yield_bound

warn: bond rows with |C| > 50, likely price-quoted series filed as yields.

In [6]:
show_check(con, 'bond_yield_bound')

,Ticker,Findings,First,Last
0,10YDEP,579,2024-03-21,2026-07-02
1,10YFRP,579,2024-03-21,2026-07-02
2,15YDEP,579,2024-03-21,2026-07-02
3,2YDEP,579,2024-03-21,2026-07-02
4,2YFRP,579,2024-03-21,2026-07-02
5,30YDEP,579,2024-03-21,2026-07-02
6,30YFRP,579,2024-03-21,2026-07-02
7,3YDEP,579,2024-03-21,2026-07-02
8,4YDEP,579,2024-03-21,2026-07-02
9,5YDEP,579,2024-03-21,2026-07-02


,SrcTable,Ticker,Date,Key,Details
0,market_data,10YCAP,2024-03-21,,C=97.84
1,market_data,10YCAP,2024-03-22,,C=98.38
2,market_data,10YCAP,2024-03-25,,C=98.11
3,market_data,10YCAP,2024-03-26,,C=97.7
4,market_data,10YCAP,2024-03-27,,C=98.12
5,market_data,10YCAP,2024-03-28,,C=98.47
6,market_data,10YCAP,2024-04-01,,C=94.92
7,market_data,10YCAP,2024-04-02,,C=94.7
8,market_data,10YCAP,2024-04-03,,C=94.67
9,market_data,10YCAP,2024-04-04,,C=95.15


### extreme_return

warn: daily |log return| > ln 2 with no same-day split; single-day glitches tagged `(spike-and-revert)`.

In [7]:
show_check(con, 'extreme_return')

,Ticker,Findings,First,Last
0,CODA,54,2001-07-24,2012-03-27
1,SMTI,45,1994-08-25,2007-08-28
2,WKSP,45,2001-06-29,2021-02-17
3,SNGX,42,1994-07-18,2026-04-28
4,QUBT,41,2007-12-14,2018-12-06
5,RCAT,40,2002-03-25,2020-12-28
6,SOBR,40,2009-06-11,2024-12-16
7,ARWR,37,1994-05-06,2016-11-30
8,LIXT,37,2008-06-12,2023-02-07
9,ABVC,35,2005-06-20,2021-11-01


,SrcTable,Ticker,Date,Key,Details
0,prices,AACG,2018-08-27,,ret=-70.1%
1,prices,AACG,2019-04-09,,ret=107.8%
2,prices,AACG,2021-02-04,,ret=951.5%
3,prices,AACG,2021-02-05,,ret=-55.7%
4,prices,AACG,2022-01-04,,ret=114.3%
5,prices,AACG,2025-08-01,,ret=-51.4%
6,prices,AACIW,2026-06-18,,ret=-53.8%
7,prices,AAME,2021-02-05,,ret=117.6%
8,prices,AAPL,2000-09-29,,ret=-51.9%
9,prices,AARD,2026-03-02,,ret=-56.2%


### stale_run

warn: 20+ consecutive identical closes.

In [8]:
show_check(con, 'stale_run')

,Ticker,Findings,First,Last
0,KELYB,84,1985-07-16,2023-11-09
1,MAYS,62,1987-02-25,2026-02-04
2,SIEB,62,1980-07-01,1997-12-29
3,HNRG,56,1994-06-07,2009-08-07
4,HYFT,56,2002-04-19,2016-12-30
5,FLUT,48,2003-02-25,2019-07-10
6,VLGEA,46,1973-05-07,1995-02-24
7,BCAL,45,2006-02-07,2022-12-29
8,EDUC,45,1980-05-30,1999-06-29
9,SKE,42,2002-03-19,2016-02-03


,SrcTable,Ticker,Date,Key,Details
0,prices,AACIU,2026-06-10,,25 identical closes 2026-05-06 .. 2026-06-10 (C=10.079999923706055)
1,prices,AACIW,2026-05-22,,51 identical closes 2026-03-12 .. 2026-05-22 (C=0.6499000191688538)
2,prices,AAME,1982-04-30,,21 identical closes 1982-04-01 .. 1982-04-30 (C=4.25)
3,prices,ABAT,2016-05-12,,48 identical closes 2016-03-07 .. 2016-05-12 (C=7.5)
4,prices,ABEV,1997-04-15,,26 identical closes 1997-03-10 .. 1997-04-15 (C=0.48750001192092896)
5,prices,ABEV,1997-06-06,,32 identical closes 1997-04-23 .. 1997-06-06 (C=0.46875)
6,prices,ABEV,1999-05-20,,27 identical closes 1999-04-14 .. 1999-05-20 (C=0.30000001192092896)
7,prices,ABEV,1999-10-26,,59 identical closes 1999-08-04 .. 1999-10-26 (C=0.27708300948143005)
8,prices,ABEV,2000-02-22,,29 identical closes 2000-01-11 .. 2000-02-22 (C=0.32499998807907104)
9,prices,ABEV,2000-06-09,,30 identical closes 2000-04-28 .. 2000-06-09 (C=0.3583329916000366)


### date_gap

warn: more than 30 calendar days between consecutive rows.

In [9]:
show_check(con, 'date_gap')

,Ticker,Findings,First,Last
0,WHLRL,11,2023-06-07,2026-05-08
1,PCG-PI,6,2008-07-11,2026-06-18
2,PW-PA,5,2015-04-01,2026-06-18
3,GLOP-PA,4,2023-09-07,2024-06-03
4,GLOP-PB,4,2023-09-07,2024-06-03
5,GLOP-PC,4,2023-09-07,2024-06-03
6,PCG-PC,4,2014-07-18,2026-06-18
7,CAPNR,2,2008-11-10,2026-06-08
8,BK,1,1973-01-10,1973-01-10
9,DHCNL,1,2026-06-18,2026-06-18


,SrcTable,Ticker,Date,Key,Details
0,prices,BK,1973-01-10,,gap 35 days after 1972-12-06
1,prices,CAPNR,2008-11-10,,gap 45 days after 2008-09-26
2,prices,CAPNR,2026-06-08,,gap 4960 days after 2012-11-08
3,prices,DHCNL,2026-06-18,,gap 34 days after 2026-05-15
4,prices,FARM,1973-01-18,,gap 38 days after 1972-12-11
5,prices,FVNNR,2026-04-22,,gap 33 days after 2026-03-20
6,prices,GLOP-PA,2023-09-07,,gap 55 days after 2023-07-14
7,prices,GLOP-PA,2023-12-07,,gap 91 days after 2023-09-07
8,prices,GLOP-PA,2024-03-07,,gap 91 days after 2023-12-07
9,prices,GLOP-PA,2024-06-03,,gap 88 days after 2024-03-07


### nonpositive_dividend

error: Dividend <= 0.

In [10]:
show_check(con, 'nonpositive_dividend')

*clean*

### oversized_dividend

warn: Dividend > 25% of the same-day close.

In [11]:
show_check(con, 'oversized_dividend')

,Ticker,Findings,First,Last
0,DEC,13,2021-03-04,2023-11-30
1,AIV,4,2020-11-03,2026-06-04
2,MXE,4,1997-12-23,2008-12-23
3,ABNY,3,2024-08-07,2025-05-01
4,ATRO,3,2001-11-14,2008-10-02
5,CANG,3,2022-05-24,2022-11-25
6,GF,3,1998-11-12,2021-12-29
7,GYRE,3,2015-08-20,2023-01-13
8,GYRO,3,2012-12-17,2016-06-16
9,KF,3,2007-11-29,2021-12-23


,SrcTable,Ticker,Date,Key,Details
0,dividends,AACG,2018-08-27,,Dividend=6.0 C=2.009999990463257 (298.5%)
1,dividends,ABNY,2024-08-07,,Dividend=3.9735 C=15.171299934387207 (26.2%)
2,dividends,ABNY,2024-11-14,,Dividend=4.316 C=15.70989990234375 (27.5%)
3,dividends,ABNY,2025-05-01,,Dividend=3.01 C=11.5 (26.2%)
4,dividends,ACI,2022-10-21,,Dividend=6.85 C=21.079999923706055 (32.5%)
5,dividends,AD,2025-08-20,,Dividend=23.0 C=53.65999984741211 (42.9%)
6,dividends,AD,2026-06-11,,Dividend=11.0 C=40.11000061035156 (27.4%)
7,dividends,AIV,2020-11-03,,Dividend=1.093178 C=3.59415602684021 (30.4%)
8,dividends,AIV,2025-10-16,,Dividend=2.23 C=5.550000190734863 (40.2%)
9,dividends,AIV,2026-02-27,,Dividend=1.45 C=4.409999847412109 (32.9%)


### invalid_split_ratio

error: Ratio <= 0 or exactly 1.

In [12]:
show_check(con, 'invalid_split_ratio')

*clean*

## Fundamentals

### balance_identity

warn: Total Liabilities + Total Equity vs Total Assets, tolerance max(1%, $100k).

In [13]:
show_check(con, 'balance_identity')

,Ticker,Findings,First,Last
0,ROIC,88,,
1,AY,76,,
2,ETRN,60,,
3,VIA,54,,
4,EVA,53,,
5,ARC,50,,
6,THMO,49,,
7,VOXX,48,,
8,ENLC,44,,
9,PETQ,36,,


,SrcTable,Ticker,Date,Key,Details
0,balance,OMEX,,us|A|2017|Q4|true,liab+equity: lhs=18356320 rhs=2972426 diff=517.55%
1,balance,OMEX,,us|A|2017|Q4|false,liab+equity: lhs=18342803 rhs=2972426 diff=517.1%
2,balance,OMEX,,us|A|2019|Q4|false,liab+equity: lhs=29335676 rhs=5329720 diff=450.42%
3,balance,OMEX,,us|A|2019|Q4|true,liab+equity: lhs=29298337 rhs=5329720 diff=449.72%
4,balance,OMEX,,us|A|2021|Q4|true,liab+equity: lhs=45327595 rhs=8908887 diff=408.79%
5,balance,OMEX,,us|A|2021|Q4|false,liab+equity: lhs=45316106 rhs=8908887 diff=408.66%
6,balance,OMEX,,us|A|2018|Q4|false,liab+equity: lhs=24794201 rhs=5473170 diff=353.01%
7,balance,OMEX,,us|A|2018|Q4|true,liab+equity: lhs=24790846 rhs=5473170 diff=352.95%
8,balance,OMEX,,us|A|2022|Q4|false,liab+equity: lhs=57484489 rhs=13281836 diff=332.81%
9,balance,OMEX,,us|A|2022|Q4|true,liab+equity: lhs=57200799 rhs=13870827 diff=312.38%


### balance_subtotals

warn: current + noncurrent vs total, assets and liabilities sides.

In [14]:
show_check(con, 'balance_subtotals')

,Ticker,Findings,First,Last
0,ROIC,158,,
1,CUTR,102,,
2,AAMC,88,,
3,ALIM,70,,
4,IVAC,70,,
5,TELL,64,,
6,HIBB,63,,
7,VIA,62,,
8,CSSE,60,,
9,MSTR,54,,


,SrcTable,Ticker,Date,Key,Details
0,balance,MULG,,us|A|2017|Q4|false,cur+noncur liab: lhs=5102679578364 rhs=19072015 diff=26754700.57%
1,balance,MULG,,us|A|2019|Q4|true,cur+noncur liab: lhs=1855308708229 rhs=16578228 diff=11191136.53%
2,balance,MULG,,us|A|2020|Q4|false,cur+noncur liab: lhs=1426101170347 rhs=22573645 diff=6317449.38%
3,balance,AY,,us|A|2020|Q4|true,cur+noncur liab: lhs=8808549966021 rhs=8191390547 diff=107434.24%
4,balance,AY,,us|Q|2020|Q4|true,cur+noncur liab: lhs=8808549966021 rhs=8191390547 diff=107434.24%
5,balance,AY,,us|Q|2021|Q4|false,cur+noncur liab: lhs=7955239529071 rhs=7999415626 diff=99347.76%
6,balance,AY,,us|A|2021|Q4|false,cur+noncur liab: lhs=7955239529071 rhs=7999415626 diff=99347.76%
7,balance,DSKE,,us|A|2015|Q4|false,cur+noncur liab: lhs=7185197230 rhs=7377460 diff=97293.92%
8,balance,DSKE,,us|A|2015|Q4|true,cur+noncur liab: lhs=7185196541 rhs=7392623 diff=97094.14%
9,balance,IGXT,,us|A|2007|Q4|true,cur+noncur liab: lhs=798076418 rhs=1059405 diff=75232.51%


### income_chain

warn: Rev+COGS=GP; GP+OtherOpInc+OpEx=OI; OI+NonOp=Pretax Adj. All additive (expenses stored negative).

In [15]:
show_check(con, 'income_chain')

,Ticker,Findings,First,Last
0,TAST,253,,
1,HA,242,,
2,MRO,182,,
3,SWN,163,,
4,BBQ,136,,
5,MSTR,129,,
6,DSKE,127,,
7,PRFT,102,,
8,ASXC,100,,
9,WRK,100,,


,SrcTable,Ticker,Date,Key,Details
0,income,ULNV,,us|A|2021|FY|true,oi+nonop: lhs=914452331 rhs=-25383 diff=3602717.23%
1,income,ULNV,,us|A|2021|FY|false,oi+nonop: lhs=914454810 rhs=-25458 diff=3592113.55%
2,income,ULNV,,us|Q|2019|Q4|true,oi+nonop: lhs=71838091 rhs=-9974 diff=720353.57%
3,income,ULNV,,us|Q|2019|Q4|false,oi+nonop: lhs=71838184 rhs=-9999 diff=718553.69%
4,income,SLNM,,us|Q|2018|Q4|true,rev+cogs: lhs=-2281167 rhs=-535 diff=426286.36%
5,income,SLNM,,us|Q|2018|Q4|false,rev+cogs: lhs=-2274208 rhs=-535 diff=424985.61%
6,income,TAST,,us|A|2015|FY|false,oi+nonop: lhs=15717000 rhs=3998 diff=393021.56%
7,income,TAST,,us|A|2015|FY|true,oi+nonop: lhs=15717000 rhs=4008 diff=392040.72%
8,income,ULNV,,us|Q|2021|Q4|false,oi+nonop: lhs=281339795 rhs=103769 diff=271021.24%
9,income,ULNV,,us|Q|2021|Q4|true,oi+nonop: lhs=281339540 rhs=103810 diff=270913.91%


### cashflow_identity

warn: op+inv+fin (+FX, disc ops) vs Net Change in Cash.

In [16]:
show_check(con, 'cashflow_identity')

,Ticker,Findings,First,Last
0,VIA,54,,
1,ULNV,37,,
2,ARC,33,,
3,EMKR,31,,
4,ASXC,26,,
5,HIBB,24,,
6,NMRD,23,,
7,TUP,23,,
8,TAST,22,,
9,NBSE,21,,


,SrcTable,Ticker,Date,Key,Details
0,cashflow,AUGX,,us|Q|2020|Q1|false,op+inv+fin: lhs=-5379638 rhs=-214 diff=2513749.53%
1,cashflow,AUGX,,us|A|2019|FY|false,op+inv+fin: lhs=1715377 rhs=224 diff=765693.3%
2,cashflow,DOMA,,us|A|2019|FY|false,op+inv+fin: lhs=125753111 rhs=-66010 diff=190606.15%
3,cashflow,DM,,us|Q|2020|Q3|false,op+inv+fin: lhs=-37333648 rhs=26638 diff=140251.84%
4,cashflow,AUGX,,us|Q|2020|Q3|false,op+inv+fin: lhs=-1979577 rhs=1535 diff=129062.67%
5,cashflow,BHVN,,us|Q|2022|Q1|false,op+inv+fin: lhs=-36106616 rhs=31000 diff=116572.95%
6,cashflow,BHVN,,us|Q|2022|Q1|true,op+inv+fin: lhs=-36064006 rhs=31000 diff=116435.5%
7,cashflow,SLNM,,us|A|2006|FY|true,op+inv+fin: lhs=433314 rhs=388 diff=111578.87%
8,cashflow,SLNM,,us|A|2017|FY|true,op+inv+fin: lhs=-128416 rhs=-116 diff=110603.45%
9,cashflow,TGAN,,us|Q|2019|Q2|false,op+inv+fin: lhs=-1480019 rhs=-1389 diff=106452.84%


### hard_invariants

error: shares <= 0 on any statement; negative Total Assets.

In [17]:
show_check(con, 'hard_invariants')

,Ticker,Findings,First,Last
0,LZB,4,,


,SrcTable,Ticker,Date,Key,Details
0,balance,LZB,,us|A|2005|Q4|false,Total Assets=-30704000
1,balance,LZB,,us|A|2003|Q4|true,Total Assets=-16308000
2,balance,LZB,,us|A|2002|Q4|true,Total Assets=-31607000
3,balance,LZB,,us|A|2005|Q4|true,Total Assets=-56180000


### negative_revenue

warn: Revenue < 0; can be genuine for financials.

In [18]:
show_check(con, 'negative_revenue')

,Ticker,Findings,First,Last
0,REEMF,32,,
1,GGROU,22,,
2,BENF,13,,
3,BRACU,13,,
4,CRSP,13,,
5,ONTX,13,,
6,UNF,13,,
7,ENO,12,,
8,HOMR,12,,
9,SAM,12,,


,SrcTable,Ticker,Date,Key,Details
0,income,A,,us|Q|2013|Q4|false,Revenue=-1170000000
1,income,A,,us|Q|2013|Q4|true,Revenue=-1170000000
2,income,AACI,,us|Q|2021|Q3|false,Revenue=-1969
3,income,AAGH,,us|Q|2022|Q4|true,Revenue=-53776
4,income,AAMC,,us|A|2023|FY|false,Revenue=-17337601
5,income,AAMC,,us|Q|2023|Q3|true,Revenue=-10559270
6,income,AAMC,,us|Q|2024|Q1|true,Revenue=-216011
7,income,AAMC,,us|Q|2022|Q3|false,Revenue=-439918
8,income,AAMC,,us|Q|2021|Q3|false,Revenue=-1296207
9,income,AAMC,,us|Q|2023|Q4|false,Revenue=-8227168


### quarters_vs_fy

warn: sum of 4 quarters vs FY for Revenue, Net Income and operating cash.

In [19]:
show_check(con, 'quarters_vs_fy')

,Ticker,Findings,First,Last
0,USLM,50,,
1,VIA,42,,
2,ELSE,29,,
3,MRO,29,,
4,BRCM,27,,
5,AEYE,26,,
6,LQMT,26,,
7,GIFI,24,,
8,HSON,24,,
9,MG,24,,


,SrcTable,Ticker,Date,Key,Details
0,income,AVYA,,us|A|2021|FY|true,Net Income: Q1..Q4=-61999951 FY=-13 diff=476922600.0%
1,income,CRSP,,us|A|2021|FY|false,Net Income: Q1..Q4=-758465397339 FY=377661 diff=200832433.06%
2,income,CRSP,,us|A|2019|FY|false,Net Income: Q1..Q4=-102004826142 FY=66858 diff=152569465.14%
3,income,CREG,,us|A|2015|FY|false,Revenue: Q1..Q4=-24544881605500 FY=24355500 diff=100777672.23%
4,income,CRSP,,us|A|2021|FY|false,Revenue: Q1..Q4=-899799384037 FY=914963 diff=98342807.2%
5,income,LQMT,,us|A|2012|FY|true,Revenue: Q1..Q4=-620998940 FY=650 diff=95538398.46%
6,income,LVVR,,us|A|2018|FY|false,Net Income: Q1..Q4=3911462756126 FY=-4273259 diff=91533581.97%
7,income,LVVR,,us|A|2018|FY|true,Net Income: Q1..Q4=3771223520741 FY=-4273259 diff=88251795.5%
8,income,LQMT,,us|A|2012|FY|true,Net Income: Q1..Q4=11065975999 FY=-14025 diff=78901889.65%
9,income,LVVR,,us|A|2018|FY|false,Revenue: Q1..Q4=-25512797437492 FY=33966599 diff=75111527.66%


## Triage notes

- (none yet)